# Initialisation

## Import modules

In [ ]:
# to render the plots inside the notebook, must be used before import matplotlib.pyplot
# %matplotlib inline

# pip install ipympl for widget see module_avac.py TODO

import datetime
import fnmatch
import glob
from linecache                      import getline
import numpy as np
from numpy                          import ma
import os
from os.path                        import expanduser
import pandas as pd
from pathlib                        import Path
# from pylab                          import *
from rasterio.features              import rasterize
from rasterio.transform             import from_bounds
import re
import shutil
import subprocess
import sys
from tabulate                       import tabulate
import tarfile
from tqdm                           import tqdm
from tqdm.notebook                  import tqdm as tqdm_nb
import yaml

# matplotlib
import matplotlib.pyplot as plt
import matplotlib
import matplotlib.colors as mcolors
from matplotlib                     import colors
from matplotlib.animation           import FuncAnimation

# scipy
from scipy.interpolate              import RegularGridInterpolator
from scipy.interpolate              import griddata

# spatial data
from osgeo                          import gdal
import shapely
from shapely.affinity               import translate

# jupyter
from jupyprint                      import jupyprint  
from IPython.display                import display
from IPython.display                import FileLink
from IPython.display                import FileLinks
from IPython.display                import HTML
from IPython.display                import Image
from IPython.display                import Markdown
from IPython.display                import Math
from IPython.display                import Video
import webbrowser

# various packages
import warnings
warnings.filterwarnings("ignore")
 
#                                                                 To ignore pylance warnings...
from clawpack.amrclaw               import region_tools           # type: ignore
from clawpack.geoclaw               import dtopotools             # type: ignore
from clawpack.geoclaw               import marching_front         # type: ignore
from clawpack.geoclaw               import fgmax_tools            # type: ignore
from clawpack.geoclaw               import fgout_tools            # type: ignore
# from clawpack.geoclaw               import topotools              # type: ignore
from clawpack.geoclaw               import topotools as topo      # type: ignore
from clawpack.geoclaw               import util                   # type: ignore
from clawpack.pyclaw                import solution as solution   # type: ignore
from clawpack.pyclaw.solution       import Solution               # type: ignore
from clawpack.visclaw               import animation_tools        # type: ignore
from clawpack.visclaw               import colormaps              # type: ignore
from clawpack.visclaw               import frametools             # type: ignore
from clawpack.visclaw               import geoplot                # type: ignore
from clawpack.visclaw               import gridtools              # type: ignore
from clawpack.visclaw               import plottools              # type: ignore
from clawpack.visclaw.plottools     import pcolorcells            # type: ignore

from module_avac                    import avac_install
from module_avac                    import avac_parameters_export
from module_avac                    import avac_parameters_import
from module_avac                    import avac_reload
from module_avac                    import claw_check_installation
from module_avac                    import claw_check_version
from module_avac                    import claw_export_dem
from module_avac                    import claw_export_dem_window
from module_avac                    import claw_export_initiation_file
from module_avac                    import correctingFactorElevation
from module_avac                    import correctingFactorSlope
from module_avac                    import create_cross_section
from module_avac                    import dict_flatten
from module_avac                    import export_profile
from module_avac                    import make_output
# from module_avac                    import raster_check
# from module_avac                    import raster_determine_type
from module_avac                    import raster_plot_topo
# from module_avac                    import raster_read_features
# from module_avac                    import raster_read_file
from module_avac                    import raster_to_topotools
from module_avac                    import rename_output_directory
from module_avac                    import shapefile_import_initial_condition
from module_avac                    import shapefile_import_polylines
from module_avac                    import WindowSelector
# from module_avac import export_raster
# from module_avac import make_animation
# from module_avac import fn_eta
# from module_avac import fn_ground
# from module_avac import fn_h

## Variables

In [ ]:
language = 'French'

# Clawpack
file_format = 'binary'             # format for files fortq*

# Latex
plt.rcParams['font.family'] = 'serif'
# plt.rcParams['text.latex.preamble'] = r'\usepackage{libertine}'  # require 2 GB package...
plt.rcParams['text.latex.preamble'] = r'\usepackage{charter}'
plt.rcParams['font.size']   = 12
plt.rcParams['text.usetex'] = True

# Figures
figure_export_params = dict(dpi = 300, bbox_inches = 'tight')

now = datetime.datetime.now()

## Directories

In [ ]:
home          = expanduser("~")      # home directory
dir_current   = Path().cwd()
dir_config    = dir_current
dir_proj      = dir_current.parent
dir_result    = dir_current / '_output'  # TODO
dir_export    = dir_proj / 'Export'
dir_images    = dir_proj / "Figures"
dir_topo      = dir_proj / "Topo"
dir_animation = dir_images / 'Animation'
dir_video     = dir_images / 'Vidéos'

text_creation = {'French':'Création du répertoire', 'English':'Creating directory'}[language]
text_warning  = {'French':'Attention: Le répertoire suivant est manquant : ', 
                 'English':'Be careful. Missing directory'}[language]

if not dir_export.exists():
     dir_export.mkdir(parents=True, exist_ok=True)
     print(f"{text_creation} {dir_export}")
if not dir_images.exists():
     dir_images.mkdir(parents=True, exist_ok=True)
     print(f"{text_creation} {dir_images}")
if not dir_animation.exists():
     dir_animation.mkdir(parents=True, exist_ok=True)
     print(f"{text_creation} {dir_animation}")
if not dir_video.exists():
     dir_video.mkdir(parents=True, exist_ok=True)
     print(f"{text_creation} {dir_video}")
if not dir_topo.exists():
     print(f"{text_warning} {dir_topo}")
     dir_topo.mkdir(parents=True, exist_ok=True)
     print(f"{text_creation} {dir_topo}")

## Real world topo?

In [ ]:
real_world_topography = True
initiation_via_qinit  = False if real_world_topography else True
# Synchronise le Makefile avec le choix de topographie
with open('config.mk', 'w') as f:
    f.write(f'REAL_WORLD_TOPO = {"1" if real_world_topography else "0"}\n')

# Parameters

## List available .asc and .shp in topo directory

In [ ]:
files_raster = list(dir_topo .glob('*.asc'))
files_shape  = list(dir_topo .glob('*.shp'))

files_raster_names = [raster.name for raster in files_raster]
files_shape_names  = [shapef.name for shapef in files_shape]

print("* Raster files in current directory:")
for file in files_raster:
    print(file.name)
print()
print("* Shapefiles in current directory:")
for file in files_shape:
    print(file.name)

## Define

In [ ]:
# definition of the computational parameters
release     = {
                'areas':                'ZA.shp',
                'correction_slope':     True,
                'correction_elevation': True,
                'd0':                   2.0,
                'gradient_hypso':       0.0003,     # correction elevation [-] (1 cm / 100 m = 0.0001)
                'nu':                   0.2,        # correction slope (Quervain model)
                'period_return':        300,
                'theta_cr':             30,         # correction slope (Quervain model)
                'z_ref':                2000,       # correction elevation
                }
rheology    = {
                'beta':                 0.,
                'C':                    0,          # cohesion
                'model':                'Voellmy',
                'mu':                   0.2,        # Coulomb / Voellmy
                'rho':                  300,
                'u_cr':                 0,
                'xi':                   1400,       # Voellmy
                }
output      = {
                'delta_t':              1,
                'language':             'French',
                'output_directory':     f"_output{release['period_return']}",
                'output_format':        'binary32',
                'verbosity':            0,
                }
objects     = { 'line':                 'profil.shp'}
gauges      = { 'gauge_record':         False,
                'gauges':               None,
                }
topography  = {
                'dem':                  'topo2m_c.asc',
                'finer_dem':            'topo1m_simple.asc',
                # 'topo_directory':       str(dir_topo),
                'topo_refinement':      False,
                'topo_source':          'real_world' if real_world_topography else 'virtual_topo',
                }
if real_world_topography: 
    computation = {
                'boundary':             'extrap',
                'cell_size':            2,
                'cfl_max':              1,
                'cfl_target':           0.5,
                'dry_limit':            0.0001,
                'force_stop':           False,
                'initial_mass':         True,
                'limiter':              'vanleer',
                'mass_frac_stop':       0.01,
                'mass_threshold_velocity':0.01,
                'max_iter':             100000,
                'nb_simul':             150,
                'output_directory':     '_output',
                'refinement':           1,
                't_max':                150,
                'topo_dir':             str(dir_topo),
                'track_mass':           True,
                }
else:
    computation = {
                'boundary_west':        'extrap',
                'boundary_east':        'extrap',
                'boundary_north':       'extrap',
                'boundary_south':       'extrap',
                'cfl_max':              0.5,
                'cfl_target':           0.25,
                'dry_limit':            1e-6,
                'dx':                   0.05,
                'dy':                   0.1,
                'force_stop':           False,
                'initial_mass':         True,
                'limiter':              'vanleer',      # 'none', 'superbee','mc','vanleer', 'minmod'
                'mass_frac_stop':       0.01,
                'mass_threshold_velocity':0.01,
                'max_iter':             100000, 
                'nb_simul':             60,
                'output_directory':     '_output',
                'refinement':           1,
                't_max':                30,
                'track_mass':           True,
                'xlower':               0.0,
                'xupper':               250,
                'ylower':               -.5,
                'yupper':               0.5,
                } 

animation   = {
                'animation_directory':  str(dir_video),
                'label_step':           500,
                'making_html':          False,
                'n_out':                computation['nb_simul'],
                'variable':             'depth',
                }
date        = { 'date':                 now}

# Warnings
if not topography['dem'] in files_raster_names: 
    print(f"Be careful: selected DEM {topography['dem']} is not in the working directory")

if not release['areas'] in files_shape_names and release['areas'] is not None: 
    print(f"Be careful: selected Shapefile {release['areas']} is not in the working directory")

# language of the notebook
language = output['language']

## Save

In [ ]:
parameters  = {
                'animation':    animation,
                'computation':  computation,
                'date':         date,
                'objects':      objects,
                'output':       output,
                'release':      release,
                'rheology':     rheology,
                'topography':   topography,
                }

file_name = 'AVAC_parameters.yaml'
avac_parameters_export(file_name, parameters)
print("Data has been written to " + file_name)

## Load & check

In [ ]:
# Importing parameter file and checking consistency
# avac_reload()
# from module_avac import avac_parameters_import
avac_parameters = avac_parameters_import('AVAC_parameters.yaml')

# Pre-processing

## Clawpack

In [ ]:
# Claw path and version
CLAW = claw_check_installation()

claw_version = claw_check_version(CLAW)

print('-- General information --')
print(f"* Clawpack version {claw_version[0]}.{claw_version[1]} installed")
if claw_version and claw_version[0] is not None and claw_version[1] is not None:
    if not (claw_version[0] == 5 and claw_version[1] >= 11):
        print("Error! This script may not work. Consider upgrading...")
    else:
        print("* Clawpack up to date.")
else:
    print("Error: Could not determine Clawpack version.")
print('* Claw directory: ', CLAW)
print('* Working directory: ', dir_current)
print('* Configuration file directory: ', dir_config)
print('* Clawpack output directory: ', dir_result )
print('* Language for plots: ', language)

avac_install(verbosity = False)

In [ ]:
# avac_reload()
# from module_avac import avac_parameters_import
# from module_avac import export_profile
# from module_avac import make_output
# from module_avac import shapefile_import_polylines
# from module_avac import WindowSelector

## DEM

### Check

In [ ]:
if raster_check(dir_topo / avac_parameters['topography']['dem']):

    xmin, xmax, ymin, ymax, nbx, nby, cell_size, dictionary_domain, topo_file_type, _, _, _ = raster_read_features(dir_topo / avac_parameters['topography']['dem'])

    print()
    print('Successful import.')
    if topo_file_type == 'claw':
        print(f"-> File {avac_parameters['topography']['dem']} is a native clawpack format")
    else:
        print(f"-> Note that file type is {topo_file_type} and is not compatible with clawpack.")
        print("   It will be converted later.")
else:
    print('Failed attempt to import the raster file')

### Load DEM & profile

In [ ]:
print(f"Fichier topo : {avac_parameters['topography']['dem']}" if language == 'French' else f"File: {avac_parameters['topography']['dem']}" )
dem_topo = raster_read_file(dir_topo / avac_parameters['topography']['dem'])

# Elevation 2D array
altitude = dem_topo.Z

# Mask NaN values
dem_topo_masked = np.ma.masked_invalid(altitude)

# Profile
profile_exists = False
if avac_parameters['objects']['line'] is not None:
    line, profile_coords = shapefile_import_polylines(dir_topo / avac_parameters['objects']['line'])
    if profile_coords is not None and len(profile_coords) > 0:
        profile_x = [pt[0] - xmin for pt in profile_coords]  # relative to graph origin
        profile_y = [pt[1] - ymin for pt in profile_coords]
        profile_exists = True

### Plot hillshade & profile

In [ ]:
# avac_reload()
# from module_avac import raster_plot_topo

fig, ax, x0, y0 = raster_plot_topo(dem_topo, grid = 200)

# Profile
if profile_exists:
    ax.plot(profile_x, profile_y, color = 'red')

### Plot slopes

In [ ]:
# Compute gradients in the x and y directions, ignoring NaN values
dem_gradient_x, dem_gradient_y = np.gradient(dem_topo_masked, cell_size)

# Compute slope as the magnitude of the gradient vector
dem_slope = np.sqrt(dem_gradient_x**2 + dem_gradient_y**2)
dem_slope_deg = np.rad2deg(np.arctan(dem_slope) )  # conversion radians -> degrees

# Create a density plot (heatmap)
fig, ax = plt.subplots(figsize = (12, 8))
plt.imshow(dem_slope_deg, cmap = 'terrain', origin = 'lower', extent = (xmin-xmin, xmax-xmin, ymin-ymin, ymax-ymin))
if hasattr(fig.canvas, 'header_visible'):
    setattr(fig.canvas, 'header_visible', False)

if language == 'French':
    plt.colorbar(label='pente (degrés)', shrink = 0.7)
    plt.xlabel(r'axe $x$ [m]')
    plt.ylabel(r'axe $y$ [m]')
    #plt.title('Pente du terrain naturel (en degrés)')
else:
    plt.colorbar(label='slope (deg)', shrink = 0.7)
    plt.xlabel(r'$x$-axis [m]')
    plt.ylabel(r'$y$-axis [m]')
    # plt.title('Density plot of ground slope (in deg)')

print(f"Origin point: (xmin = {xmin:0.2f}, ymin = {ymin:0.2f}) ")

# Profile
if profile_exists:
    ax.plot(profile_x, profile_y, color = 'black')

plt.show()

In [ ]:
fig.savefig(dir_images / 'pente_naturelle.png', dpi = figure_export_params['dpi'], bbox_inches = figure_export_params['bbox_inches'])

### Plot cross-section

In [ ]:
if not profile_exists:
    profile_coords = [[0, 5], [250, 5]]
 
dem1 = dem_topo_masked
dem2 = dem_slope_deg

# Create a grid of (x, y) coordinates for the DEM
x_coords = np.linspace(xmin, xmax, dem1.shape[1])  # shape[1] = number of cols
y_coords = np.linspace(ymin, ymax, dem1.shape[0])  # shape[0] = number of rows

# Create cross-section (elevation)
distances, elevations, points = create_cross_section(dem1, x_coords, y_coords, profile_coords)  # pyright: ignore[reportPossiblyUnboundVariable]

legend_1 = {'French': 'altitude', 'English':'elevation'}
legend_2 = {'French': 'pente',    'English':'slope'}

fig = plt.figure(figsize = (12, 6))
ax1 = plt.subplot(1, 1, 1)
ax1.plot(distances, elevations, '-', color = 'brown', linewidth = 1, label = legend_1[language])

if language == 'French':
    ax1.set_ylabel(f"altitude (m)")
    ax1.set_xlabel(f"distance depuis l'origine (m)")
else:
    ax1.set_ylabel(f"Elevation (m)")
    ax1.set_xlabel(f"Distance along profile (m)")

#plt.title('Topographic Cross-Section')
plt.grid(True, alpha = 0.3)

# Secondary axis
ax2 = ax1.twinx()   
#ax2.set_ylim((0,C_max))
if language == 'French':
    ax2.set_ylabel(f'pente (°)' )
else:
    ax2.set_ylabel(f'Slope (°)' )

# Create cross-section (slope)
distances, pentes, points = create_cross_section(dem2, x_coords, y_coords, profile_coords)  # pyright: ignore[reportPossiblyUnboundVariable]
ax2.plot(distances, pentes, '--', color = 'black', linewidth = 1, label = legend_2[language])

ax2.tick_params(axis = 'y', labelcolor = 'black')
ax1.text(0.04, 0.91, '(b)', horizontalalignment = 'center', verticalalignment = 'center', transform = ax1.transAxes, fontsize = 13)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc = 'best', ncol = 2, frameon = False, bbox_to_anchor = (0.5, 0.85))
plt.show()
fig.savefig(dir_images / 'profil_pente.png', dpi = figure_export_params['dpi'], bbox_inches = figure_export_params['bbox_inches'])

In [ ]:
export_profile(dir_export / 'profil_TN.txt', distances, elevations)
if language == 'English':
    print(f"slope min={dem_slope_deg.min():.2f}°  max={dem_slope_deg.max():.2f}°  mean={dem_slope_deg.mean():.2f}°")
    print(f"Export of the terrain profile to {str(dir_export / 'profil_TN.txt')}")
else:
    print(f"Pente du terrain naturel : min = {dem_slope_deg.min():.2f}°,  max = {dem_slope_deg.max():.2f}°, et moyenne = {dem_slope_deg.mean():.2f}°")
    print(f"Export du profil de terrain vers {str(dir_export / 'profil_TN.txt')}")
 

## Release areas

### Load

In [ ]:
areas           = avac_parameters['release']['areas']
corr_elev       = avac_parameters['release']['correction_elevation']
corr_slope      = avac_parameters['release']['correction_slope']
d_0             = avac_parameters['release']['d0']
gradient_hypso  = avac_parameters['release']['gradient_hypso']
nu              = avac_parameters['release']['nu']
return_period   = avac_parameters['release']['period_return']
theta_cr        = avac_parameters['release']['theta_cr']
z_ref           = avac_parameters['release']['z_ref']

starting_polygons, nb_areas = shapefile_import_initial_condition(dir_topo / areas)

# Enumerating the polygons forming the starting areas
polygon_za = list(starting_polygons.geometry)

### Plot areas

In [ ]:
fig, ax, x0, y0 = raster_plot_topo(dem_topo, grid = 200)
starting_polygons_translated = starting_polygons.copy()

# Apply translation to each geometry
starting_polygons_translated.geometry = starting_polygons_translated.geometry.apply(
    lambda geom: translate(geom, xoff = -x0, yoff = -y0))
starting_polygons_translated.plot(ax = ax)

# Profile
if profile_exists:
    ax.plot(profile_x, profile_y, color='red')


### Initial snow depth

In [ ]:
# Old code
# from tqdm import tqdm

# # Computing the initial release depth
# h_0               = avac_parameters['release']['d0']
# xi                = dem_topo.X[0,:]
# yi                = dem_topo.Y[:,0]
# zi                = np.ones((len(yi), len(xi))) * h_0  # initial condition
# z_ref             = avac_parameters['release']['z_ref']
# gradient_hypso    = avac_parameters['release']['gradient_hypso']
# theta_cr          = avac_parameters['release']['theta_cr']
# nu                = avac_parameters['release']['nu']
# return_period     = avac_parameters['release']['period_return']

# # data out of the starting zone are set to 0
# # runtime: duration may exceed 1'
# for i in tqdm(range(len(xi))):
#       for j in range(len(yi)):
#             pt = shapely.geometry.Point([xi[i], yi[j]])
#             if not np.any(pt.within(polygon_za)): 
#                   zi[j, i] = 0
#             else:
#                   if avac_parameters['release']['correction_elevation'] and not avac_parameters['release']['correction_slope']:
#                         zi[j, i] = zi[j, i] + correctingFactorElevation(altitude[j, i], z_ref, gradient_hypso)
#                   elif avac_parameters['release']['correction_elevation'] and avac_parameters['release']['correction_slope']:
#                         zi[j, i] = (zi[j, i] + correctingFactorElevation(altitude[j, i], z_ref, gradient_hypso)) * correctingFactorSlope(dem_slope[j, i], theta_cr, nu)
#                   elif not avac_parameters['release']['correction_elevation'] and avac_parameters['release']['correction_slope']:
#                         zi[j, i] = zi[j, i] * correctingFactorSlope(dem_slope[j, i], theta_cr, nu)

In [ ]:
xi              = dem_topo.X[0,:]
yi              = dem_topo.Y[:,0]
delta           = cell_size / 2

# Create a spatial transform mapping grid to coordinates
transform       = from_bounds(xi.min() - delta, yi.max() + delta, xi.max() + delta, yi.min() - delta, len(xi), len(yi))

# Burn geometry into a binary true/false mask matching
geoms           = polygon_za if isinstance(polygon_za, (list, tuple)) else [polygon_za]
rasterized_out  = rasterize(geoms, out_shape = (len(yi), len(xi)), transform = transform, default_value = 1, fill = 0)
if rasterized_out is not None:
    in_polygon_mask = rasterized_out.astype(bool)
else:
    in_polygon_mask = np.zeros((len(yi), len(xi)), dtype=bool)

# Vectorized assignment and calculations
depths = np.where(in_polygon_mask, d_0, 0.0)
if corr_elev:
    depths = np.where(in_polygon_mask, depths + correctingFactorElevation(altitude, z_ref, gradient_hypso), depths)
if corr_slope:
    depths = np.where(in_polygon_mask, depths * correctingFactorSlope(dem_slope, theta_cr, nu), depths)

### Plot depths

In [ ]:
masked_zi = np.ma.masked_where(depths == 0, depths)
z_lower_bound = int(np.nanmin(dem_topo.Z))
z_upper_bound = int(np.nanmax(dem_topo.Z))

fig, (ax1, ax2) = plt.subplots(2, 1)
fig.set_figheight(10)
fig.set_figwidth(15)

# ax1 = Map
ax1.set_aspect('equal')

if language == 'French':
    ax1.set_xlabel(r'axe $x$')
    ax1.set_ylabel(r'axe $y$')
else:
    ax1.set_xlabel(r'$x$-axis')
    ax1.set_ylabel(r'$y$-axis')

ax1.contour(dem_topo.X-xmin, dem_topo.Y-ymin, ma.masked_where(dem_topo.Z < z_lower_bound, dem_topo.Z), 
            levels=range(z_lower_bound,z_upper_bound,5), colors='black', alpha=0.5, linewidths=.4)

print(f"Origin point: (xmin = {xmin:0.2f}, ymin = {ymin:0.2f}) ")
pc = ax1.imshow(masked_zi,origin='lower',extent = [xmin-xmin,xmax-xmin,ymin-ymin,ymax-ymin],cmap='jet_r')
cb = plt.colorbar(pc, ax=ax1, label=r'$d_0$ (m)', shrink=0.8)  # colorbar normale
ax1.text(0.04, 0.9, '(a)', horizontalalignment='center', verticalalignment='center', transform=ax1.transAxes, fontsize=13)

# ax2 = Histogramme
# Ajuster la position de ax2 pour qu'elle corresponde à la largeur effective de ax1
pos1 = ax1.get_position()
pos2 = ax2.get_position()
# Calculer la largeur effective de ax1 (incluant l'espace pris par la colorbar)
effective_width = pos1.width
new_x2 = pos1.x0  # Même position x que ax1
ax2.set_position([new_x2, pos2.y0, effective_width, pos2.height])

style = {'facecolor': 'orange', 'edgecolor': 'black', 'linewidth': 0.5}
data_points = depths[depths > 0]
total_points = len(data_points)
weights = np.ones_like(data_points) / total_points
ax2.hist(data_points, bins=50, weights=weights, **style)
ax2.set_xlabel(r'$d_0$ (m)')
ax2.set_ylabel(r'$\mathrm{prob}(d)$')
ax2.text(0.04, 0.9, '(b)', horizontalalignment='center', verticalalignment='center', transform=ax2.transAxes, fontsize=13)
ax2.grid(alpha = 0.5)

plt.show()

In [ ]:
fig.savefig(dir_images / f'zone_départ_{return_period}ans.png', dpi = figure_export_params['dpi'], bbox_inches = figure_export_params['bbox_inches'])

### Text check

In [ ]:
# Initial conditions
print(f'Initial condition features for T = {return_period} yr:')
display(Math(fr"\bullet\text{{ Total surface of the starting zones }}  S = {(starting_polygons.area.sum()/1e4):0.2f}  \,\text{{ha}}"))
display(Math(fr"\bullet\text{{ Total volume mobilized }}  V = {(depths.sum() * cell_size**2 / 1e3):0.1f} \times10^3\,\text{{m}}^3"))
display(Math(fr"\bullet\text{{ Initial reference depth at }} {z_ref} \text{{ m, }} h_0 = {d_0} \text{{ m}}"))
display(Math(fr"\bullet\text{{ Release depth range }} d_{{min}} = {depths[depths>0].min():.2f} \text{{ m, }} d_{{max}} = {depths.max():.2f} \text{{ m}}. "))
display(Math(fr"\bullet\text{{ Mean value }}  d_{{mean}} = {100*depths[depths>0].mean():.0f} \text{{ cm}}. "))
display(Math(fr"\bullet\text{{ Maximum release depth at }} z = {int(altitude[depths==depths.max()][0])} \text{{ m, }} d_0 = {depths.max():0.2f} \text{{ m}}"))

## Export required files

In [ ]:
# for active development
# avac_reload()
# from module_avac import claw_export_dem
# from module_avac import claw_export_dem_window
# from module_avac import claw_export_initiation_file
# from module_avac import raster_check
# from module_avac import raster_determine_type
# from module_avac import raster_read_features
# from module_avac import raster_read_file
# from module_avac import shapefile_import_polylines
# from module_avac import WindowSelector

### Initial conditions

In [ ]:
claw_export_initiation_file(dem_topo, depths)

### DEM

In [ ]:
file_type = 
print(f"Le fichier topo pour le calcul des avalanche est {avac_parameters['topography']['dem']}")
print(f"Ce fichier est de type {topo_file_type}.")
if topo_file_type != 'claw':
    print(f"Attention : le fichier topographique n'est pas compatible avec geoclaw !")
    print("Conversion vers le format clawpack et transfert vers topography.asc...")
    file_name                               = 'topo_AVAC.asc'
    topography['dem']                       = file_name
    avac_parameters['topography']['dem']    = file_name
    claw_export_dem(xmin, xmax, ymin, ymax, nbx, nby, dem_topo.Z, name_file = str(dir_topo / file_name))

### DEM refinement (if any)

In [ ]:
if avac_parameters['topography']['topo_refinement']:
    if raster_check(avac_parameters['topography']['finer_dem']):
        topo_file_fine = raster_read_file(avac_parameters['topography']['finer_dem'])
        print()
        print('Successful import.')
        xmin_fine, xmax_fine, ymin_fine, ymax_fine, nbx_fine, nby_fine, cell_size_fine, _, _, dictionary_domain_fine, _, _ = raster_read_features(avac_parameters['topography']['finer_dem'])
        altitude_fine = topo_file_fine.Z
    else:
        print('Failed attempt to import the raster file')
else:
    print('No topography refinement')

In [ ]:
if avac_parameters['topography']['topo_refinement']:
    # %matplotlib widget must be used at the top of the notebook
    if avac_parameters['topography']['topo_refinement']:
        # window = (969752,6539330,970000,6539430)
        sel = WindowSelector(dem_topo, grid = 200)

In [ ]:
if avac_parameters['topography']['topo_refinement'] and sel.window is not None:    
    window = list(map(int, sel.window))
    file_name = 'topo_AVAC_fine.asc'
    claw_export_dem_window(topo_file_fine, window, name_file = file_name)
    print(f"Successful export to {file_name}.")
    xmin_fine, xmax_fine, ymin_fine, ymax_fine, nbx_fine, nby_fine, cell_size_fine, dictionary_domain_refined, _, _, _, _ = raster_read_features(file_name)

### Gauges (if any)

In [ ]:
gauges_dict = {
                'gauge_recording':  False,
                'gauges':           [],
                }

if avac_parameters['objects']['line'] is not None:
    ligne, coords_array = shapefile_import_polylines(dir_topo / avac_parameters['objects']['line'])
    if coords_array is not None and len(coords_array) > 0:
        gauge_x, gauge_y = coords_array[0] 
        # gauge_x, gauge_y = 965780, 6536190
        gauge_1 = [1, float(gauge_x), float(gauge_y)]  # float() pour compatibilité YAML safe_load
        gauges_dict = {
                'gauge_recording':  True,
                'gauges':           [gauge_1],
                }

### Configuration

In [ ]:
save_copy = False
file_name = 'AVAC_configuration.yaml'

if avac_parameters['topography']['topo_refinement']:
    refinement = {
                    'delta_t':          1,
                    'fine_dict':        dictionary_domain_refined,
                    'finer_dem':        'topography_fine.asc',
                    'topo_refinement':  True,
                    }
else:
    refinement = {
                    'delta_t':          None,
                    'fine_dict':        'None',
                    'finer_dem':        'None',
                    'topo_refinement':  False,
                    }


if initiation_via_qinit:
    topo_dictionary = {
                    'initiation_file':  None,
                    'topo_source':      topography['topo_source'],
                    'topofile':         'topography.asc',
                    'type_dem':         3,
                    'type_init':        0,
                    }
    # release['theta']            = float(theta)
    # release['xb']               = float(xb)
    release["free_surface"]     = 'parallel'
    avac_parameters['release']  = release
else:
    topo_dictionary = {
                    'initiation_file':  'init.xyz',
                    'topo_source':      topography['topo_source'],
                    'topofile':         'topography.asc',
                    'type_dem':         3,
                    'type_init':        1,
                    }  

avac_configuration = {
    'animation':    avac_parameters['animation'],
    'computation':  avac_parameters['computation'], 
    'dem_extent':   dictionary_domain,
    'file_names':   topo_dictionary,
    'gauges':       gauges_dict,
    'output':       avac_parameters['output'], 
    'refinement':   refinement,
    'release':      avac_parameters['release'],
    'rheology':     avac_parameters['rheology'], 
    }

avac_configuration_flat = dict_flatten(avac_configuration)

print("Configuration file:")
for key, value in avac_configuration_flat.items():
    var_name = key.replace('.', '_')
    globals()[var_name] = value
    print(f"* {var_name} = {value}")

avac_parameters_export(file_name, avac_configuration)
print(f"Data has been written to {file_name}.")

if save_copy:
    file_name_copy = f"AVAC_configuration{release['period_return']}.yaml"
    avac_parameters_export(file_name_copy, avac_configuration)
    print(f"Copy has been written to {file_name_copy}.")

# Running AVAC

In [ ]:
# Running AVAC
make_output(avac_parameters, verbosity = False)

In [ ]:
# Change output directory name?
change_output_directory_name = True
overwrite_directory          = True

rename_output_directory(avac_configuration, dir_current, change_output_directory_name, overwrite_directory)
dir_result = str(dir_current / avac_parameters['output']['output_directory'])

In [ ]:
#reading the mass file
plt.close('all')  # ferme les figures widget avant de repasser en inline

# Plot with contour lines
%matplotlib inline
mass_evolution = pd.read_csv(Path(dir_result)/'mass.txt')

text_legend = {'French':r'Fraction du volume initial en mouvement  (en \%)',
               'English':r'Initial mass fraction still moving (en \%)'}[language]
 

fig, ax = plt.subplots(figsize=(8,4))
#ax.plot(mass_evolution['t_s'], mass_evolution['wet_mass_kg'],    label='masse dans le domaine')
ax.plot(mass_evolution['t_s'], 100 * mass_evolution['moving_mass_kg'] / mass_evolution['wet_mass_kg'])
ax.set_xlabel('$t$ (s)')

ax.set_ylabel(rf'{text_legend}')
ax.legend()
ax.grid()
fig.savefig(dir_images / "evolution_avalanche_mass.png", dpi = figure_export_params['dpi'])

In [ ]:
# all in once
# if the DEM and init.xyz are already existing, it is possible to run AVAC directly
all_in_once = False
if all_in_once:
    avac_parameters = avac_parameters_import('AVAC_parameters.yaml')
    avac_configuration = {
        'animation':    avac_parameters['animation'],
        'computation':  avac_parameters['computation'],
        'dem_extent':   dictionary_domain,
        'file_names':   avac_parameters['topography'],
        # no gauges?
        'output':       avac_parameters['output'],
        # no refinement?
        # no release?
        'rheology':     avac_parameters['rheology'], 
        }
    avac_configuration_flat = dict_flatten(avac_configuration)
    print("Configuration file:")
    for key, value in avac_configuration_flat.items():
        var_name = key.replace('.', '_')
        globals()[var_name] = value
        print(f"* {var_name} = {value}")

    file_name = 'AVAC_configuration.yaml'
    avac_parameters_export(file_name, avac_configuration)
    print(f"Data has been written to {file_name}.")
    
    make_output(avac_parameters, verbosity=False)

# Solutions

## Loading the solution (fgmax) and defining the color maps

In [ ]:
outdir = avac_parameters['computation']['output_directory']
t_files = glob.glob(outdir + '/fort.t0*')
times = []
for f in t_files:
    lines = open(f,'r').readlines()
    for line in lines:
        if 'time' in line: 
            t = float(line.split()[0])
    times.append(t)
times.sort()
print('Output times found: ',times)

In [ ]:
# Read fgmax data:
outdir = avac_parameters['output']['output_directory']
fgno = 1
fg = fgmax_tools.FGmaxGrid()
fg.read_fgmax_grids_data(fgno)
fg.read_output(outdir = outdir)

if avac_parameters['topography']['topo_refinement']:
    fgno    = 2
    fg_zoom = fgmax_tools.FGmaxGrid()
    fg_zoom.read_fgmax_grids_data(fgno)
    fg_zoom.read_output(outdir = outdir)

In [ ]:
# Fields to be computed
velocity            = ma.masked_where(fg.h<0.0001, fg.s )
depths               = ma.masked_where(fg.h<0.0001, fg.h )
rho                 = avac_parameters['rheology']['rho']
pressure            = 0.5 * rho * velocity**2 / 1e3
pressure.fill_value = -9999
depths.fill_value    = -9999
velocity.fill_value = -9999

In [ ]:
ϵ = 1e-6 # minimum value
#depth
bounds_depth = np.array([ϵ,0.1,0.5,1,2,4,8,16])
cmap_depth   = mpl.colors.ListedColormap([[.9,.9,1],[.6,.6,1],\
                 [.3,.3,1],[0,0,1], [1,.8,.8],\
                 [1,.6,.6], [1,0,0]])

# pressure (kPa)
bounds_pressure = np.array([ϵ,1e3,5e3,1e4,3e4,5e4,1e5,2e5])/1e3
cmap_pressure   = mpl.colors.ListedColormap([[.9,.9,1],[.6,.6,1],\
                 [.3,.3,1],[0,0,1], [1,.8,.8],\
                 [1,.6,.6], [1,0,0]])

# velocity norm
bounds_speed = np.array([ϵ,0.5,1,2,4,8,16,32])
cmap_speed   = mpl.colors.ListedColormap([[.9,.9,1],[.6,.6,1],\
                 [.3,.3,1],[0,0,1], [1,.8,.8],\
                 [1,.6,.6], [1,0,0]])

# Set color for value exceeding top of range to purple if needed:
cmap_speed.set_over(color=[1,0,1])
cmap_depth.set_over(color=[1,0,1])
cmap_pressure.set_over(color=[1,0,1])

# Set color for land points with too low values to light green if needed:
cmap_speed.set_under(color=[.7,1,.7])
cmap_depth.set_under(color=[.7,1,.7])
cmap_pressure.set_under(color=[.7,1,.7])

# Normalization
norm_speed = colors.BoundaryNorm(bounds_speed, cmap_speed.N)
norm_depth = colors.BoundaryNorm(bounds_depth, cmap_depth.N)
norm_pressure = colors.BoundaryNorm(bounds_pressure, cmap_pressure.N)

## Plot on contours

In [ ]:
z_lower_bound = int(np.nanmin(dem_topo.Z))
z_upper_bound = int(np.nanmax(dem_topo.Z))

In [ ]:
# mini snaphots of (h,p,v)
plt.close('all')  # Ferme les figures widget avant de repasser en inline
# Plot with contour lines
%matplotlib inline
fig, (ax1, ax2, ax3) = plt.subplots(1, 3)
fig.set_figheight(5)
fig.set_figwidth(20)

# Set aspect ratio for EACH subplot individually
ax1.set_aspect('equal')
ax2.set_aspect('equal')
ax3.set_aspect('equal')

ax1.contour(dem_topo.X-xmin, dem_topo.Y-ymin, ma.masked_where(dem_topo.Z<z_lower_bound, dem_topo.Z), \
            levels=range(z_lower_bound,z_upper_bound,10), colors='grey',alpha=0.5,linewidth=.1) 
pc1 = pcolorcells(fg.X-xmin, fg.Y-ymin, pressure, cmap=cmap_pressure, norm=norm_pressure,ax=ax1)

ax2.contour(dem_topo.X-xmin, dem_topo.Y-ymin, ma.masked_where(dem_topo.Z<z_lower_bound, dem_topo.Z), \
            levels=range(z_lower_bound,z_upper_bound,10), colors='grey',alpha=0.5)
pc2 = pcolorcells(fg.X-xmin, fg.Y-ymin, depths, cmap=cmap_depth, norm=norm_depth,ax=ax2)

ax3.contour(dem_topo.X-xmin, dem_topo.Y-ymin, ma.masked_where(dem_topo.Z<z_lower_bound, dem_topo.Z), \
            levels=range(z_lower_bound,z_upper_bound,10), colors='grey',alpha=0.5)
pc3 = pcolorcells(fg.X-xmin, fg.Y-ymin, velocity, cmap=cmap_speed, norm=norm_speed,ax=ax3)

if language == 'French':
    ax1.set_title('Pression cinétique maximale (kPa)', fontsize=14, color='black')
    ax2.set_title("Hauteur maximale d'écoulement (m)", fontsize=14, color='black')
    ax3.set_title('Vitesse maximale (m/s)', fontsize=14, color='black')
    cb1 = plt.colorbar(pc1, extend='max', shrink=0.7,label='pression (kPa)')
    cb2 = plt.colorbar(pc2, extend='max', shrink=0.7,label='hauteur (m)')
    cb3 = plt.colorbar(pc3, extend='max', shrink=0.7,label='vitesse (m/s)')
else:
    ax1.set_title('Maximum kinetic pressure (kPa)', fontsize=14, color='black')
    ax2.set_title('Maximum depth (m)', fontsize=14, color='black')
    ax3.set_title('Maximum velocity (m/s)', fontsize=14, color='black')
    cb1 = plt.colorbar(pc1, extend='max', shrink=0.7,label='pressure (kPa)')
    cb2 = plt.colorbar(pc2, extend='max', shrink=0.7,label='depth (m)')
    cb3 = plt.colorbar(pc3, extend='max', shrink=0.7,label='velocity (m/s)')

print(f"Origin point: (xmin = {xmin:0.2f}, ymin = {ymin:0.2f})")

In [ ]:
return_period = release['period_return']
fig.savefig(dir_images / f'result_max_over_contours_{return_period}yr.png', dpi = figure_export_params['dpi'], bbox_inches = figure_export_params['bbox_inches'])

## Runout

In [ ]:
z_runout = np.min(fg.B[fg.h>0.00001])
x_runout = fg.X[fg.B==z_runout] [0]
y_runout = fg.Y[fg.B==z_runout] [0]
display(Markdown(f"Runout position (point of furthest reach): $x = {x_runout:.2f}$ m and $y = {y_runout:.2f}$."))
print(f"Runout elevation (elevation of the point of furthest reach): {z_runout:.2f} m.")

## profiles of maximum depth, pressure and velocity

In [ ]:
if avac_parameters['objects']['line'] is not None:
    ligne, coords_array = shapefile_import_polylines(dir_topo / avac_parameters['objects']['line'])
    x1,y1 = coords_array[0]
    x2,y2 = coords_array[-1]
else:
    from maketopo import fnc_plane
    x1 = 0
    y1 = 0
    z1 = fnc_plane(x1, y1, theta ,0)
    x2 = 200
    y2 = 0
    z2 = fnc_plane(x2, y2, theta ,0)
    return_period = 0


In [ ]:
avac_reload()
from module_avac import raster_plot_topo

In [ ]:
# cross-section
fig, axes, x0, y0 = raster_plot_topo(dem_topo,grid = 500,ylabel_position="left")
# puis ajouter ce qu'on veut sur ax :
axes.plot([x1 - x0, x2 - x0], [y1 - y0, y2 - y0], color='black',linewidth=1.5)
plt.tight_layout(pad=0.4)

print(f"Origin point: (xmin = {x0:0.2f}, ymin = {y0:0.2f}) ")
 
pc = pcolorcells(fg.X-x0, fg.Y-y0, depths, cmap=cmap_depth, norm=norm_depth, ax=axes)
cb = plt.colorbar(pc, extend='max', shrink=0.7)
if language == 'French':
    cb.set_label(r'hauteur (m)')
else:
    cb.set_label('maximum depth (m)')

fig.savefig(dir_images / f"carte_hauteur_T{return_period}.png", dpi = figure_export_params['dpi'], bbox_inches = figure_export_params['bbox_inches'])

In [ ]:
# cross-section
fig, axes, x0, y0 = raster_plot_topo(dem_topo,grid = 500,ylabel_position="left")
# puis ajouter ce qu'on veut sur ax :
axes.plot([x1 - x0, x2 - x0], [y1 - y0, y2 - y0], color='black',linewidth=1.5)
plt.tight_layout(pad=0.4)

 
print(f"Origin point: (xmin = {x0:0.2f}, ymin = {y0:0.2f}) ")
 
pc = pcolorcells(fg.X-x0, fg.Y-y0, pressure, cmap=cmap_pressure, norm=norm_pressure, ax=axes)
cb = plt.colorbar(pc, extend='max', shrink=0.7)

if language == 'French':
    cb.set_label(r'pression maximale (kPa)')
else:
    cb.set_label('maximum pressure (Kpa)')
 
fig.savefig(dir_images / f"carte_pression_T{return_period}.png", dpi = figure_export_params['dpi'], bbox_inches = figure_export_params['bbox_inches'])

In [ ]:
avac_reload()
from module_avac import create_cross_section

In [ ]:
# Profile along a line
# x_coords, y_coords: coordinate arrays for the DEM grid
# profile_coords: coordinates extracted from the polyline

# Get coordinates from the polyline

if avac_parameters['objects']['line'] is not None:
    profile_coords = list(ligne.geometry.iloc[0].coords)
else:
    profile_coords = [[x1,y1,0],[x2,y2,0]]  
# plotting the deposit cross-section

dem1 = fg.B.T
dem2 = fg.h.T

# Create a grid of (x, y) coordinates for the DEM
x_coords = np.linspace(xmin, xmax, dem1.shape[1])
y_coords = np.linspace(ymin, ymax, dem1.shape[0])

# Create cross-section
distances, elevations, points = create_cross_section(dem1, x_coords, y_coords, profile_coords)
distances, flow_depth, _      = create_cross_section(dem2, x_coords, y_coords, profile_coords)

# Plot the cross-section
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
plt.plot(distances, elevations, '-',color = 'brown', linewidth=2)
plt.plot(distances, elevations+flow_depth, 'b-', linewidth=1)
plt.fill_between(distances, elevations+flow_depth, elevations, color='lightskyblue')
plt.xlabel('Distance along profile (m)')
plt.ylabel('Elevation (m)')
#plt.title('Topographic Cross-Section')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Get coordinates from  polyline
# profile_coords = list(ligne.geometry.iloc[0].coords)
# plotting the deposit cross-section

dem_v = velocity.T
#dem_p = pressure.T
dem_p = (velocity*velocity/1e3/2*rho ).T
dem_h = depths.T
# Create a grid of (x, y) coordinates for the DEM
x_coords = np.linspace(xmin, xmax, dem1.shape[1])
y_coords = np.linspace(ymin, ymax, dem1.shape[0])
distances, velocity_profile, _ = create_cross_section(dem_v, x_coords, y_coords, profile_coords)
_, depth_profile, _            = create_cross_section(dem_h, x_coords, y_coords, profile_coords)
_, pressure_profile, _         = create_cross_section(dem_p, x_coords, y_coords, profile_coords)

# Create cross-sections
fig, ((ax1, ax2, ax3)) = plt.subplots(3, 1)
fig.set_figheight(12)
fig.set_figwidth(12)
# Plot the velocity profile
ax1.plot(distances, velocity_profile, 'b-', linewidth=1)
if language == 'French':   
    ax1.set_ylabel("vitesse (m/s)")
else:
    ax1.set_ylabel("Velocity (m/s)")
 

# Plot the depth profile
ax2.plot(distances, depth_profile, 'b-', linewidth=1)
if language == 'French':
    ax2.set_ylabel("hauteur (m)")
else:
    ax2.set_ylabel("Depth (m)")
 
# Plot the pressure profile
ax3.plot(distances, pressure_profile, 'b-', linewidth=1)
if language == 'French':
    ax3.set_xlabel("Distance depuis le point origine (m)")
    ax3.set_ylabel("pression (kPa)")
else:
    ax3.set_xlabel("Distance along the cross-section (m)")
    ax3.set_ylabel("Pressure (kPa)")
 
 

axes_list = (ax1,ax2,ax3)
étiquette = {str(ax1):'(a)',str(ax2):'(b)',str(ax3):'(c)'}
for ax in axes_list:
    ax.text(0.05, 0.91,  étiquette[str(ax)] ,
            horizontalalignment='center',
            verticalalignment='center',
            transform = ax.transAxes,fontsize=13)
    ax.grid(True, alpha=0.3)

plt.show()
fig.savefig(dir_images / f'profil_{return_period}ans.png', dpi = figure_export_params['dpi'], bbox_inches = figure_export_params['bbox_inches'])



In [ ]:
export_profile(dir_export / f'profil_vitesse_{return_period}.txt',distances, velocity_profile)
export_profile(dir_export / f'profil_hauteur_{return_period}.txt',distances, depth_profile)
export_profile(dir_export / f'profil_pression_{return_period}.txt',distances, pressure_profile)

print(f"pression finale : p = {pressure_profile[0]:.1f} kPa")
print(f"vitesse finale : v = {velocity_profile[0]:.1f} m/s")
print(f"hauteur finale : h = {depth_profile[0]:.1f} m")

## Zoom on refinement

In [ ]:
if avac_parameters['topography']['topo_refinement']:


    velocity_zoom = ma.masked_where(fg_zoom.h<0.001, fg_zoom.s )
    depth_zoom    = ma.masked_where(fg_zoom.h<0.001, fg_zoom.h )
    rho        = avac_parameters['rheology']['rho']
    pressure_zoom = 0.5*rho*velocity_zoom**2/1e3
    pressure_zoom.fill_value = -9999
    depth_zoom.fill_value    = -9999
    velocity_zoom.fill_value = -9999


In [ ]:
if avac_parameters['topography']['topo_refinement']:
    # cross-section
    topo_file_zoom = raster_read_file('topography_fine.asc')

    fig, axes, x0, y0 = raster_plot_topo(topo_file_zoom,grid = 50,ylabel_position="left")
    # puis ajouter ce qu'on veut sur ax :
    axes.plot([x1 - x0, x2 - x0], [y1 - y0, y2 - y0], color='black',linewidth=1.5)
    axes.set_xlim([dictionary_domain_refined['xmin']-x0,dictionary_domain_refined['xmax']-x0])
    axes.set_ylim([dictionary_domain_refined['ymin']-y0,dictionary_domain_refined['ymax']-y0])
    plt.tight_layout(pad=0.4)

    
    if language == 'French':
        cb.set_label('hauteur maximale (m)')
    else:
        cb.set_label('maximum depth (m)')

    print(f"Origin point: (xmin = {x0:0.2f}, ymin = {y0:0.2f}) ")
    
    pc = pcolorcells(fg_zoom.X-x0, fg_zoom.Y-y0, depth_zoom, cmap=cmap_depth, norm=norm_depth, ax=axes,alpha = 0.5)
    cb = plt.colorbar(pc, extend='max', shrink=0.7)
    cb.set_label(r'hauteur maximale (m)' if language == 'French' else r'maximum depth (m)')
    fig.savefig(dir_images /"zoom_depth.png", dpi = figure_export_params['dpi'], bbox_inches = figure_export_params['bbox_inches'])

In [ ]:
if avac_parameters['topography']['topo_refinement']:

    fig, axes, x0, y0 = raster_plot_topo(topo_file_zoom,grid = 50,ylabel_position="left")
    # puis ajouter ce qu'on veut sur ax :
    axes.plot([x1 - x0, x2 - x0], [y1 - y0, y2 - y0], color='black',linewidth=1.5)
    axes.set_xlim([dictionary_domain_refined['xmin']-x0,dictionary_domain_refined['xmax']-x0])
    axes.set_ylim([dictionary_domain_refined['ymin']-y0,dictionary_domain_refined['ymax']-y0])
    plt.tight_layout(pad=0.4)

    if language == 'French':
        cb.set_label('pression cinétique maximale (kPa)')
    else:
        cb.set_label('maximum kinetic pressure (m)')

    print(f"Origin point: (xmin = {x0:0.2f}, ymin = {y0:0.2f}) ")
    
    pc = pcolorcells(fg_zoom.X-x0, fg_zoom.Y-y0, pressure_zoom, cmap=cmap_pressure, norm=norm_pressure, ax=axes,alpha = 0.5)
    cb = plt.colorbar(pc, extend='max', shrink=0.7)
    cb.set_label(r'pression cinétique maximale (kPa)' if language == 'French' else r'maximum kinetic pressure (kPa)')
    fig.savefig(dir_images /"zoom_pressure.png", dpi = figure_export_params['dpi'], bbox_inches = figure_export_params['bbox_inches'])

In [ ]:
if avac_parameters['topography']['topo_refinement']:
    # Get coordinates from your polyline
    profile_coords = list(ligne.geometry.iloc[0].coords)
    # plotting the deposit cross-section
    xmin_zoom = dictionary_domain_refined['xmin']
    xmax_zoom = dictionary_domain_refined['xmax']
    ymin_zoom = dictionary_domain_refined['ymin']
    ymax_zoom = dictionary_domain_refined['ymax']
    
    dem_v_zoom = velocity_zoom.T
    #dem_p = pressure.T
    dem_p_zoom = (velocity_zoom*velocity_zoom/1e3/2*rho ).T
    dem_h_zoom = depth_zoom.T
    # Create a grid of (x, y) coordinates for the DEM
    x_coords = np.linspace(xmin_zoom, xmax_zoom, dem_v_zoom.shape[1])
    y_coords = np.linspace(ymin_zoom, ymax_zoom, dem_v_zoom.shape[0])
    distances, velocity_profile_zoom, _ = create_cross_section(dem_v_zoom, x_coords, y_coords, profile_coords)
    _, depth_profile_zoom, _            = create_cross_section(dem_h_zoom, x_coords, y_coords, profile_coords)
    _, pressure_profile_zoom, _         = create_cross_section(dem_p_zoom, x_coords, y_coords, profile_coords)

    # Create cross-sections
    fig, ((ax1, ax2, ax3)) = plt.subplots(3, 1)
    fig.set_figheight(12)
    fig.set_figwidth(12)
    # Plot the velocity profile
    ax1.plot(distances, velocity_profile_zoom, 'b-', linewidth=1)
    if language == 'French':
        
        ax1.set_ylabel("vitesse (m/s)")
    else:
        ax1.set_xlabel("Distance along the cross-section (m)")
        ax1.set_ylabel("Velocity (m/s)")
    
    # Plot the depth profile
    ax2.plot(distances, depth_profile_zoom, 'b-', linewidth=1)
    if language == 'French':
        #ax2.set_xlabel("distance depuis le point origine (m)")
        ax2.set_ylabel("hauteur (m)")
    else:
        #ax2.set_xlabel("Distance along the cross-section (m)")
        ax2.set_ylabel("Depth (m)")
    
    # Plot the pressure profile
    ax3.plot(distances, pressure_profile_zoom, 'b-', linewidth=1)
    if language == 'French':
        ax3.set_xlabel("Distance depuis le point origine (m)")
        ax3.set_ylabel("pression (kPa)")
    else:
        ax3.set_xlabel("Distance along the cross-section (m)")
        ax3.set_ylabel("Pressure (kPa)")
    
    axes_list = (ax1,ax2,ax3)
    étiquette = {str(ax1):'(a)',str(ax2):'(b)',str(ax3):'(c)'}
    for ax in axes_list:
        ax.text(0.05, 0.91,  étiquette[str(ax)] ,
                horizontalalignment='center',
                verticalalignment='center',
                transform = ax.transAxes,fontsize=13)
        ax.grid(True, alpha=0.3)


    plt.show()
    fig.savefig(dir_images /f'profil_{return_period}ans_zoom.png', dpi = figure_export_params['dpi'], bbox_inches = figure_export_params['bbox_inches'])



In [ ]:
avac_parameters

# Export to Qgis

In [ ]:
# Exporting the rasters for Qgis
if avac_parameters['topography']['topo_source']=='real_world':
    if language == 'French':
        print(f"Export des rasters vers {dir_export}") 
    else:
        print(f"Exporting raster files tp {dir_export}") 
    export_raster(dir_export / f'depth_max_{return_period}yr.asc',depths.filled(),xmin,ymin,avac_parameters['computation']['cell_size'],ndata=-9999)
    export_raster(dir_export / f'pressure_max_{return_period}yr.asc',pressure.filled(),xmin,ymin,avac_parameters['computation']['cell_size'],ndata=-9999)
    export_raster(dir_export / f'velocity_max_{return_period}yr.asc',velocity.filled(),xmin,ymin,avac_parameters['computation']['cell_size'],ndata=-9999)

    if avac_parameters['topography']['topo_refinement']:
        if language == 'French':
            print(f"Export des rasters de la zone raffinée vers {dir_export}") 
        else:
                print(f"Exporting refined raster files tp {dir_export}") 
        export_raster(dir_export / f'depth_zoom_max_{return_period}yr.asc',depth_zoom.filled(),dictionary_domain_refined['xmin'],
                    dictionary_domain_refined['ymin'],dictionary_domain_refined['cell_size'],ndata=-9999)
        export_raster(dir_export / f'pressure_zoom_max_{return_period}yr.asc',pressure_zoom.filled(),dictionary_domain_refined['xmin'],
                    dictionary_domain_refined['ymin'],dictionary_domain_refined['cell_size'],ndata=-9999)
        export_raster(dir_export / f'velocity_zoom_max_{return_period}yr.asc',velocity_zoom.filled(),dictionary_domain_refined['xmin'],
                    dictionary_domain_refined['ymin'],dictionary_domain_refined['cell_size'],ndata=-9999)

# Make animation (two-dimensional plot)

In [ ]:
avac_reload()
from module_avac import make_animation

In [ ]:
# Making animation
make_animation(avac_parameters,verbosity=False)


In [ ]:
if avac_parameters['animation']['making_html']:
    name_html = (f'/AVAC_animation_for_'+avac_parameters['animation']['variable']+f'_{return_period}yr.html')
    url_animation = dir_video / name_html
    print('\n Link to animation file: ', url_animation)
    webbrowser.open(str(url_animation))
path_animation = dir_video / (f'/AVAC_animation_for_'+avac_parameters['animation']['variable']+f'_{return_period}yr.mp4')

# Gauges

In [ ]:
from clawpack.pyclaw.gauges import GaugeSolution

gauge = GaugeSolution(gauge_id=1, path=outdir)

# Données disponibles après lecture :
gauge.id          # int : numéro de la jauge
gauge.location    # tuple : (x, y) de la jauge
gauge.t           # ndarray : temps
gauge.q           # ndarray(n_vars, n_times) : variables (h, hu, hv, ...)
gauge.level       # ndarray : niveau AMR utilisé à chaque instant


In [ ]:
import matplotlib.pyplot as plt
h  = gauge.q[0, :]   # hauteur
hu = gauge.q[1, :]   # quantité de mouvement x
hv = gauge.q[2, :]   # quantité de mouvement y
velocity = np.where(h>0,np.sqrt(hu**2+hv**2)/h,0)
kinetic_pressure = rheology['rho']*0.5*velocity**2
cd = 1
hydro_pressure  = cd*rheology['rho']*9.81*h/2


fig,((ax1, ax2),(ax3, ax4))   = plt.subplots(2, 2,  figsize=(10, 10))
 
# depth evolution
ax1.plot(gauge.t, h, 'black' )

if language == 'French':
    ax1.set_xlabel(r"temps $t$ (s)")
    ax1.set_ylabel(r"hauteur $h$ (m)")
else:
    ax1.set_xlabel("Time (s)")
    ax1.set_ylabel("Flow depth (m)")
ax1.grid()

# velocity evolution
ax2.plot(gauge.t, velocity,'black' )
if language == 'French':
    ax2.set_xlabel(r"temps $t$ (s)")
    ax2.set_ylabel(r"vitesse $\bar u$ (m/s)")
else:
    ax2.set_xlabel("Time (s)")
    ax2.set_ylabel("Velocity (m/s)")
ax2.grid()

# pressure evolution
ax3.plot(gauge.t, kinetic_pressure/1000, 'red', label=r"pression cinétique  $p_c$" if language=='French' else "kinetic pressure")
ax3.plot(gauge.t, hydro_pressure/1000, 'blue', label=r"pression hydrostatique  $p_h$" if language=='French' else "hydrostatic pressure")
if language == 'French':
    ax3.set_xlabel(r"temps $t$ (s)")
    ax3.set_ylabel(r"pression $p$ (kPa)")
else:
    ax3.set_xlabel("Time (s)")
    ax3.set_ylabel("Pressure (kPa)")
ax3.grid()
ax3.legend()

# depth-pressuere
ax4.plot(h, kinetic_pressure/1000, 'red', label=r"pression cinétique $p_c$" if language=='French' else "kinetic pressure")
ax4.plot(h, hydro_pressure/1000,   'blue', label=r"pression hydrostatique $p_h$" if language=='French' else "hydrostatic pressure")
if language == 'French':
    ax4.set_ylabel(r"pression $p$ (kPa)")
    ax4.set_xlabel(r"hauteur $h$ (m)")
else:
    ax4.set_xlabel("Pressure (kPa)")
    ax4.set_xlabel("Flow depth (m)")


ax4.grid()
for ax, i in zip((ax1,ax2,ax3,ax4),range(4)):
    étiquette = ['(a)','(b)','(c)','(d)'][i]
    ax.text(0.05, 0.9,  étiquette  ,
        horizontalalignment='center',
        verticalalignment='center',
        transform = ax.transAxes,fontsize=13)

fig.savefig(dir_images / "Gauges_avalanches.png", dpi = figure_export_params['dpi'], bbox_inches = figure_export_params['bbox_inches'])

# Plotting the solution (at any time) from fgout

## Importing the data

In [ ]:

fgno          = 1
outdir        = avac_parameters['computation']['output_directory']
output_format = avac_parameters['output']['output_format']
fgout_grid    = fgout_tools.FGoutGrid(fgno, outdir,output_format)
fgout_grid.read_fgout_grids_data() # required as of v5.11.0

fgframe = avac_parameters['animation']['n_out'] # last time
fgout = fgout_grid.read_frame(fgframe)



## Mapping the depth

In [ ]:
# DEM and flow depth
bottom = fgout.B
depths  = ma.masked_where(fgout.h == 0, fgout.h)


opacity = 0.75
if avac_parameters['objects']['line'] is not None:
    ligne, coords_array = shapefile_import_polylines(dir_topo / avac_parameters['objects']['line'])
    x_0, y_0 = coords_array[0]
    x_1, y_1 = coords_array[-1]
else:
    x_0, y_0 = x1,y1
    x_1, y_1 = x2,y2


fig, axes, x0, y0 = raster_plot_topo(dem_topo,grid=500)
axes.plot([x_0 - x0, x_1 - x0],
                   [y_0 - y0, y_1 - y0],
                    color='black')
pc1 = pcolorcells(fgout.X-x0, fgout.Y-y0, depths, cmap=cmap_depth, norm=norm_depth,ax=axes,alpha=opacity)
cb1 = plt.colorbar(pc1, extend='max', shrink=0.7)


if language == 'French':
    cb1.set_label('hauteur finale (m)')
else:
    cb1.set_label('final depth (m)')


In [ ]:
pcolorcells(fgout.X-x0, fgout.Y-y0, bottom, cmap=cmap_depth, norm=norm_depth,ax=axes,alpha=opacity)
display(fig)

## Plotting the cross-section at final time

In [ ]:
# Plotting the cross-section at final time

fgframe = avac_parameters['animation']['n_out'] # last time
fgout = fgout_grid.read_frame(fgframe)

# fgout.B has shape (mx, my); transpose to (my, mx) for RegularGridInterpolator
dem1 = fgout.B.T  # DEM,   shape (my, mx)
dem2 = fgout.h.T  # Depth, shape (my, mx)

# Fill masked values if necessary
dem1_filled = np.ma.filled(dem1, np.nan) if np.ma.is_masked(dem1) else dem1
dem2_filled = np.ma.filled(dem2, np.nan) if np.ma.is_masked(dem2) else dem2

# Coordinate grid from fgout (not from the DEM file)
# fgout.X shape is (mx, my): x varies along axis 0, y along axis 1
x_coords = fgout.X[:, 0]  # shape (mx,)
y_coords = fgout.Y[0, :]  # shape (my,)

# Interpolators
interpolator_dem = RegularGridInterpolator(
    (y_coords, x_coords), dem1_filled, method='linear', bounds_error=False, fill_value=np.nan
)
interpolator_depth = RegularGridInterpolator(
    (y_coords, x_coords), dem2_filled, method='linear', bounds_error=False, fill_value=np.nan
)
interpolator_free_surface = RegularGridInterpolator(
    (y_coords, x_coords), dem1_filled + dem2_filled, method='linear', bounds_error=False, fill_value=np.nan
)

# Define cross-section line
num_points = 1000
x_line = np.linspace(x_0, x_1, num_points)
y_line = np.linspace(y_0, y_1, num_points)
points = np.column_stack((y_line, x_line))

# Interpolate
dem_section           = interpolator_dem(points)
depth_section         = interpolator_depth(points)
free_surface_section  = interpolator_free_surface(points)

# Compute distance (handles non-straight lines if needed)
dx = np.diff(x_line)
dy = np.diff(y_line)
segment_lengths = np.sqrt(dx**2 + dy**2)
distance        = np.concatenate(([0], np.cumsum(segment_lengths)))

# Plotting
 
fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(10, 4))
 

ax1.plot(distance, free_surface_section, 'blue', label="Avalanche Surface")
ax1.fill_between(distance, dem_section, free_surface_section, color='lightskyblue', alpha=0.3 )
ax1.plot(distance, dem_section, 'brown', label="ground")
if language == 'French':
    ax1.set_xlabel("Distance depuis le point origine (m)")
    ax1.set_ylabel("Altitude (m)")
else:
    ax1.set_xlabel("Distance along the cross-section (m)")
    ax1.set_ylabel("Elevation (m)")

print(f"Cross-Section from ({x_0:0.0f}, {y_0:0.0f}) to ({x_1:0.0f}, {y_1:0.0f}) at time {fgout.t} s")

#plt.legend()
ax1.grid()


# depth profile
ax2.plot(distance, depth_section, 'blue', label="Avalanche Surface")
ax2.grid()
if language == 'French':
    ax2.set_xlabel("Distance depuis le point origine (m)")
    ax2.set_ylabel("Épaisseur de l'avalanche (m)")
else:
    ax2.set_xlabel("Distance along the cross-section (m)")
    ax2.set_ylabel("Avalanche depth (m)")

plt.show()

## Animation: avalanche flow depth on terrain

In [ ]:


# --- Setup figure ---
fig, ax = plt.subplots(figsize=(12, 4))
ax.set_xlabel("Distance along cross-section (m)")
ax.set_ylabel("Elevation (m)")
ax.grid()

# Create all artists
dem_line, = ax.plot(distance, dem_section, 'brown', label="Terrain")
free_line, = ax.plot([], [], 'b-', label="Avalanche Surface")
depth_fill = ax.fill_between([], [], color='lightskyblue', alpha=0.5)

# Get frame information
nb_frames = avac_parameters['animation']['n_out']  # Total number of frames
frame_indices = range(nb_frames)  # Frame numbers (0 to nb_frames-1)

# Barre de progression : widget notebook — se met a jour meme sous %%capture
_pbar_eta = tqdm_nb(total=nb_frames, desc='Rendu', unit='frame')
def init():
    """Initialize animation"""
    free_line.set_data([], [])
    depth_fill.set_verts([])
    return free_line, depth_fill

def update(frame_index):
    """Update frame for given frame index"""
    try:
        fgout = fgout_grid.read_frame(frame_index + 1)  # Frame numbers start at 1
        h     = fgout.h.T
        B     = fgout.B.T
        
        # Interpolate Avalanche surface (B + h)
        free_surface = RegularGridInterpolator(
            (y_coords, x_coords), B + h,
            method='linear', bounds_error=False
        )(np.column_stack((y_line, x_line)))
        
        # Create proper closed polygon vertices
        verts = np.vstack([
            np.column_stack((distance, free_surface)),  # Avalanche surface
            [(distance[-1], dem_section[-1])],          # Bottom right
            np.flipud(np.column_stack((distance, dem_section))),  # Terrain
            [(distance[0], free_surface[0])]           # Close polygon
        ])
        
        # Update artists
        free_line.set_data(distance, free_surface)
        depth_fill.set_verts([verts])
        
        ax.set_title(f"Time = {fgout.t:.2f} seconds (Frame {frame_index + 1}/{nb_frames})")
        #ax.relim()
        #ax.autoscale_view()
        
    except Exception as e:
        print(f"Error processing frame {frame_index + 1}: {str(e)}")
        return init()
    
    _pbar_eta.update(1)  # à changer en cas de variable vitesse _pbar_velocity, _pbar_depth
    return free_line, depth_fill

# Create animation
ani = FuncAnimation(
    fig, update, frames=frame_indices,
    init_func=init, blit=True, interval=100
)

# Display with multiple fallback options
plt.close()


In [ ]:
%%capture --no-display 
# remove line above if you want to see the messages and progression
HTML(ani.to_jshtml())

## Animation: avalanche depth (with no terrain)

In [ ]:
# Animation: time variation in avalanche depth  (with no terrain)

h_max = 3 # max flow depth for y-axis

# --- Setup Figure ---
fig, ax = plt.subplots(figsize=(12, 6))
ax.set_xlabel("Distance along cross-section (m)", fontsize=12)
ax.set_ylabel("Flow Depth (m)", fontsize=12)
ax.grid(True, linestyle=':', alpha=0.7)

# Set consistent axis limits
x_min, x_max = distance.min(), distance.max()
ax.set_xlim(x_min, x_max)
ax.set_ylim(-0.05, h_max)  # Fixed y-range with h_max as upper bound

# Initialize flow depth line
flow_depth_line, = ax.plot([], [], 'b-', linewidth=2, label="Flow Depth")
ax.legend(fontsize=12)

# Add zero line reference
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)

# Barre de progression : widget notebook — se met a jour meme sous %%capture
_pbar_depth = tqdm_nb(total=nb_frames, desc='Rendu', unit='frame')

def init():
    """Initialize animation with empty data"""
    flow_depth_line.set_data([], [])
    return flow_depth_line,

def update(frame_index):
    """Update plot for time t"""
    if frame_index == 0:
        _pbar_depth.reset()   # reinitialise si 2e rendu (ex. save)
    fgout = fgout_grid.read_frame(frame_index + 1)
    
    # Extract and interpolate flow depth
    h = fgout.h.T
    depth_interp = RegularGridInterpolator(
        (y_coords, x_coords), h,
        method='linear', bounds_error=False
    )(np.column_stack((y_line, x_line)))
    
    # Update plot
    flow_depth_line.set_data(distance, depth_interp)
    
    # Dynamic title with max depth
    max_depth = np.nanmax(depth_interp)
    ax.set_title(f"Time = {fgout.t:.2f} sec | Max Depth = {max_depth:.2f} m", 
                fontsize=14, pad=20)
    
    _pbar_depth.update(1)  # à changer en cas de variable vitesse _pbar_velocity
    return flow_depth_line,

# Create animation
ani = FuncAnimation(
    fig, update, frames=range(nb_frames),
    init_func=init, blit=True, interval=100
)

# Display
plt.close()


In [ ]:
%%capture --no-display
# remove line if you want to see progression
HTML(ani.to_jshtml())

In [ ]:
%%capture --no-display
# remove line if you want to see progression
ani.save(video_dir / 'animation_flow-depth_along_path.mp4', writer='ffmpeg', fps=5)

In [ ]:
# maximum of depth h and front position as a function of time
array_maxh = []
array_t = []
array_xf = []
for frame_index in range(nb_frames):
    fgout = fgout_grid.read_frame(frame_index + 1)
    # Extract and interpolate flow depth
    h = fgout.h.T
    max_depth = np.max(h)
    array_maxh.append(max_depth)
    array_t.append(fgout.t)
    z_f = fgout.B[fgout.h>computation['dry_limit']][-1]
    x_f = fgout.X[fgout.B==z_f] [0]
    array_xf.append(x_f)

fig, (ax1,ax2) = plt.subplots(2, layout="constrained")
ax1.plot(array_t,array_maxh)
ax1.set_xlabel(r"$t$ (s)")
ax1.set_ylabel(r"$\max h$ (m)")
ax1.grid()
ax2.plot(array_t,array_xf)
ax2.grid()
ax2.set_xlabel(r"$t$ (s)")
ax2.set_ylabel(r"$x_f$ (m)")
fig.savefig(dir_images / "maxh_xf_over_time.png", dpi = figure_export_params['dpi'])

## Animation: avalanche velocity (with no terrain)

In [ ]:
u_max = 10 # max flow velocity for y-axis

# --- Setup Figure ---
fig, ax = plt.subplots(figsize=(12, 6))
ax.set_xlabel("Distance along cross-section (m)", fontsize=12)
ax.set_ylabel("Velocity (m/s)", fontsize=12)
ax.grid(True, linestyle=':', alpha=0.7)

# Set consistent axis limits
x_min, x_max = distance.min(), distance.max()
ax.set_xlim(x_min, x_max)
ax.set_ylim(-0.05, u_max)  # Fixed y-range with u_max as upper bound

# Initialize flow depth line
flow_velocity_line, = ax.plot([], [], 'b-', linewidth=2, label="Flow velocity")
ax.legend(fontsize=12)

# Add zero line reference
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
# Barre de progression : widget notebook — se met a jour meme sous %%capture
_pbar_velocity = tqdm_nb(total=nb_frames, desc='Rendu', unit='frame')
def init():
    """Initialize animation with empty data"""
    flow_velocity_line.set_data([], [])
    return flow_velocity_line,

def update(frame_index):
    """Update plot for time t"""
    fgout = fgout_grid.read_frame(frame_index + 1)
    
    # Extract and interpolate flow depth
    u = fgout.s.T
    velocity_interp = RegularGridInterpolator(
        (y_coords, x_coords), u,
        method='linear', bounds_error=False
    )(np.column_stack((y_line, x_line)))
    
    # Update plot
    flow_velocity_line.set_data(distance, velocity_interp)
    
    # Dynamic title with max depth
    max_velocity = np.nanmax(velocity_interp)
    ax.set_title(f"Time = {fgout.t:.2f} sec | Max velocity = {max_velocity:.2f} m", 
                fontsize=14, pad=20)
    _pbar_velocity.update(1)  # à changer en cas de variable vitesse _pbar_velocity
    return flow_velocity_line,

# Create animation
ani = FuncAnimation(
    fig, update, frames=range(nb_frames),
    init_func=init, blit=True, interval=100
)

# Display
plt.close()

In [ ]:
%%capture --no-display
# remove line if you want to see progression
HTML(ani.to_jshtml())

In [ ]:
%%capture --no-display
# remove line if you want to see progression
ani.save(video_dir / 'animation_flow-velocity_along_path.mp4', writer='ffmpeg', fps=5)

# MCMC

In [ ]:
mcmc_dir    = dir_proj / 'MCMC'
runs_dir    = mcmc_dir / 'Runs'
language      = avac_parameters['output']['language']
text_creation = {'French':'Création du répertoire', 'English':'Creating directory'}[language]
text_warning  = {'French':'Attention: Le répertoire suivant est manquant : ', 
                 'English':'Be careful. Missing directory'}[language]
if not mcmc_dir.exists():
     mcmc_dir.mkdir(parents=True, exist_ok=True)
     print(f"{text_creation} {mcmc_dir}")
if not runs_dir.exists():
     runs_dir.mkdir(parents=True, exist_ok=True)
     print(f"{text_creation} {runs_dir}")

In [ ]:
μ_min = avac_parameters['rheology']['mu']-0.025
μ_max = avac_parameters['rheology']['mu']+0.050
ξ_min = avac_parameters['rheology']['xi']-100
ξ_max = avac_parameters['rheology']['xi']+200
Δ_min = 0.7
Δ_max = 1.1

import random
import shutil
#dirName = 'runs'  #directory in which the results lie

# def EraseFile(repertoire):
#     files=os.listdir(repertoire)
#     for filename in files:
#         os.remove(repertoire + "/" + filename)

# try:
#     # Create target Directory
#     os.mkdir(dirName)
#     print("I have created " , dirName ,  " where all the final outcomes will lie. ") 
# except FileExistsError:
#     print("The " , dirName ,  " directory already exists.")
# EraseFile(dir_current+"/" + "runs") 

In [ ]:
if avac_parameters['topography']['topo_refinement']:
    refinement = {'fine_dict':dictionary_domain_refined,'finer_dem':'topography_fine.asc','topo_refinement':True,'delta_t':1}
else:
    refinement = {'fine_dict':'None','finer_dem':'None','topo_refinement':False,'delta_t':None}
topo_dictionary = {'topofile':'topography.asc', 'initiation_file':'init.xyz',
                       'type_dem':3,'type_init':1,'topo_directory':str(dir_topo)}  

# Opening the Loop parameters files
ParameterFile = open(mcmc_dir / 'Loop_parameters.txt','a')
# Redefining the dictionaries of parameters
new_computation = avac_parameters['computation'].copy()
new_rheology    = avac_parameters['rheology'].copy()
new_computation['refinement']     = 1
new_refinement                    = refinement.copy()
new_refinement['topo_refinement'] = False
new_release = avac_parameters['release'].copy()

# loop
for i in range(350,400):
    Δ_i = random.uniform(Δ_min,Δ_max)
    μ_i = random.uniform(μ_min,μ_max)
    ξ_i = random.uniform(ξ_min,ξ_max)
    dref = depths.copy() 
    dref = Δ_i*dref
    new_rheology['mu'] = μ_i
    new_rheology['xi'] = ξ_i
    new_release['d0']  = release['d0']*Δ_i
    avac_configuration = {
        'animation':    avac_parameters['animation'],
        'computation':  new_computation, 
        'dem_extent':   dictionary_domain,
        'file_names':   topo_dictionary,
        'gauges':       gauges_dict,
        'output':       avac_parameters['output'], 
        'refinement':   new_refinement,
        'release':      new_release,
        'rheology':     new_rheology, 
        }
    
    file_name = 'AVAC_configuration.yaml'
    avac_parameters_export(file_name, avac_configuration)
    
    # Saving the values in the Loop parameter file
    ParameterFile.write(str(μ_i)+" "+str(ξ_i)+" "+str(release['d0']*Δ_i)+"\n")
    # exporting initial condition
    claw_export_initiation_file(dem_topo, dref)
    # running clawpack
    make_output(avac_configuration,verbosity=False)
    # copy fron _output to MCMC/Runs
    shutil.copyfile(dir_result+'/gauge00001.txt', runs_dir / f'run{i}.txt')

ParameterFile.close()

In [ ]:
# Saving the results in run.txt
import pandas as pd
files_h = []
files_p = []
files_v = []
nbr = 400
for i in range(nbr):
    results = pd.read_csv(runs_dir / f'run{i}.txt',comment = '#',names=['amr','t','h','hu','hv','z'],sep=r"\s+")
    h = results['h']
    v = np.where(h>0,np.sqrt(results['hu']**2+results['hv']**2)/h,0)
    p = avac_parameters['rheology']['rho']*0.5*v**2
    t = results['t']
    files_h.append([t,h])
    files_p.append([t,p])
    files_v.append([t,v])

In [ ]:
# Read parameters to color curves
params = pd.read_csv(Path(working_directory)/'Loop_parameters.txt', sep=r"\s+")
mu_vals = params['mu'].values  # one value per run

# Normalize mu for colormap
norm_mu = plt.Normalize(mu_vals.min(), mu_vals.max())
cmap_mu = plt.cm.plasma

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(12, 12))

# Collect arrays for median envelope
all_h, all_v, all_p = [], [], []


for i in range(nbr):
    h = np.asarray(files_h[i][1])
    t = np.asarray(files_h[i][0])
    v = np.asarray(files_v[i][1])
    p = np.asarray(files_p[i][1]) / 1000
    color = cmap_mu(norm_mu(mu_vals[i]))

    ax1.plot(t, h, color=color, alpha=0.2, lw=0.6)
    ax2.plot(t, v, color=color, alpha=0.2, lw=0.6)
    ax3.plot(t, p, color=color, alpha=0.2, lw=0.6)
    ax4.plot(h, p, color=color, alpha=0.2, lw=0.6)

    all_h.append(h); all_v.append(v); all_p.append(p)

# Median envelope (interpolate on common time grid)
t_ref = np.asarray(files_h[0][0])
h_med = np.median(np.array([np.interp(t_ref, np.asarray(files_h[i][0]), np.asarray(files_h[i][1])) for i in range(nbr )]), axis=0)
v_med = np.median(np.array([np.interp(t_ref, np.asarray(files_v[i][0]), np.asarray(files_v[i][1])) for i in range(nbr )]), axis=0)
p_med = np.median(np.array([np.interp(t_ref, np.asarray(files_p[i][0]), np.asarray(files_p[i][1])/1000) for i in range(nbr )]), axis=0)

label_med = 'médiane' if language=='French' else 'median'
ax1.plot(t_ref, h_med, 'k', lw=2, label=label_med)
ax2.plot(t_ref, v_med, 'k', lw=2, label=label_med)
ax3.plot(t_ref, p_med, 'k', lw=2, label=label_med)
ax4.plot(h_med, p_med, 'k', lw=2, label=label_med)

# Labels
if language == 'French':
    ax1.set_xlabel(r"temps $t$ (s)");  ax1.set_ylabel(r"hauteur $h$ (m)")
    ax2.set_xlabel(r"temps $t$ (s)");  ax2.set_ylabel(r"vitesse $\bar u$ (m/s)")
    ax3.set_xlabel(r"temps $t$ (s)");  ax3.set_ylabel(r"pression $p_c$ (kPa)")
    ax4.set_xlabel(r"hauteur $h$ (m)"); ax4.set_ylabel(r"pression $p_c$ (kPa)")
else:
    ax1.set_xlabel("time (s)");   ax1.set_ylabel("Flow depth (m)")
    ax2.set_xlabel("time (s)");   ax2.set_ylabel("Velocity (m/s)")
    ax3.set_xlabel("time (s)");   ax3.set_ylabel("Pressure (kPa)")
    ax4.set_xlabel("Flow depth (m)"); ax4.set_ylabel("Pressure (kPa)")

for ax in (ax1, ax2, ax3, ax4):
    ax.grid(alpha=0.3)
    #ax.legend()

for ax, i in zip((ax1,ax2,ax3,ax4),range(4)):
    étiquette = ['(a)','(b)','(c)','(d)'][i]
    ax.text(0.05, 0.9,  étiquette  ,
        horizontalalignment='center',
        verticalalignment='center',
        transform = ax.transAxes,fontsize=13)

fig.savefig(mcmc_dir / "Gauges_MCMC.png", dpi = figure_export_params['dpi'], bbox_inches = figure_export_params['bbox_inches'])

In [ ]:
max_h = []
max_p = []
for i in range(nbr):
    max_p.append(np.max(files_p[i][1])/1000)
    max_h.append(list(files_h[i][1][files_p[i][1]==np.max(files_p[i][1])])[0])

In [ ]:
fig = plt.figure(figsize=(8,4))
axes = plt.subplot(1, 1, 1)
axes.scatter(max_h,max_p)

In [ ]:
fig = plt.figure(figsize=(8,4))
axes = plt.subplot(1, 1, 1)
axes.hist(max_p,density=True)

In [ ]:
fig = plt.figure(figsize=(8,4))
axes = plt.subplot(1, 1, 1)
axes.hist(max_h,density=True)

In [ ]:
import seaborn as sns
df = pd.DataFrame({'pressure': max_p, 'depth': max_h })

In [ ]:
df

In [ ]:
g = sns.JointGrid(x='depth', y='pressure', data=df) # This is a figure
g = g.plot(sns.regplot, sns.histplot,) # Draw 3 visuals on the figure
g.figure.set_size_inches(10,5)

In [ ]:
g = sns.JointGrid(data=df, x='depth', y='pressure')
g.plot(sns.scatterplot, sns.histplot,color="#207AC0")

# Grille sur le graphique central uniquement
g.ax_joint.grid(True, alpha=0.3)

# Noms des axes
g.ax_joint.set_xlabel(r"hauteur $h$ [m]")
g.ax_joint.set_ylabel(r"pression $p_c$ [kPa]")
g.figure.set_size_inches(12,6)
g.figure.savefig(mcmc_dir / "pression_max.png", dpi = figure_export_params['dpi'], bbox_inches = figure_export_params['bbox_inches'])

In [ ]:
df_params = pd.read_csv(mcmc_dir / 'Loop_parameters.txt',skiprows=1,names=['mu','xi','d0'],sep=" ")

In [ ]:
#plt.style.use('default')
from matplotlib.ticker import FuncFormatter
import numpy as np

def sci_fmt(x, _):
    if x == 0:
        return '$0$'
    exp = int(np.floor(np.log10(abs(x))))
    coef = x / 10**exp
    if abs(coef - round(coef)) < 1e-10:
        return r'$%d\times10^{%d}$' % (round(coef), exp)
    return r'$%.1f\times10^{%d}$' % (coef, exp)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(12, 4))
ax1.hist(df_params['mu'],density=True,bins=20,  color="#50A6E8")
ax2.hist(df_params['xi'],density=True,bins=20,  color="#50A6E8")
ax3.hist(df_params['d0'],density=True,bins=20,  color="#50A6E8")
from matplotlib.ticker import FuncFormatter

ax2.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{x*1e4:.0f}'))
ax2.set_ylabel(r' $(\times10^{-4})$', labelpad=-2)

ax1.set_ylabel(r"prob")
ax1.set_xlabel(r"$\mu$")
ax2.set_xlabel(r"$\xi$")
ax3.set_xlabel(r"$d_0$ (m)")
fig.savefig(mcmc_dir / "histogrammes.png", dpi = figure_export_params['dpi'], bbox_inches = figure_export_params['bbox_inches'])

In [ ]:
max(df_params['xi'][df_params['xi']<500])

In [ ]:
df_params['d0'].max()

In [ ]:
df_params['d0'].min()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist(max_p,density=True,bins=20,  color="#8AC4F1")
ax2.hist(max_h,density=True,bins=20,  color="#8AC4F1")
sns.kdeplot(data=df, x="pressure", ax=ax1, color='k', bw_adjust=0.2,
            clip=(0, None), cut=0)

sns.kdeplot(data=df, x="depth",ax=ax2,color='k',bw_adjust=0.2,clip=(0, None), cut=0)
ax1.set_ylabel(r"prob");ax2.set_ylabel(r" ")
ax1.set_xlabel(r"$p_c$ (kPa)")
ax2.set_xlabel(r"$h$ (m)")
fig.savefig(mcmc_dir / "pdf-h-p.png", dpi = figure_export_params['dpi'], bbox_inches = figure_export_params['bbox_inches'])

In [ ]:
from scipy.stats import gaussian_kde
from scipy.optimize import brentq
from scipy.integrate import quad

max_p_array = np.array(max_p)

kde_p = gaussian_kde(max_p_array, bw_method=0.2/max_p_array.std(ddof=1)*max_p_array.std(ddof=1))

total_p = quad(kde_p, 0, np.inf)[0]          # masse sur [0, +∞)
cdf_p = lambda x: quad(kde_p, 0, x)[0] / total_p

p_quantile = brentq(lambda x: cdf_p(x) - 0.9, max_p_array.min(),
                    max_p_array.max() + 5*max_p_array.std())
print(f"Quantile 90% = {p_quantile:.2f} kPa")

p_quantile = brentq(lambda x: cdf_p(x) - 0.9, max_p_array.min(),
                    max_p_array.max() + 5*max_p_array.std())
print(f"Quantile 90% = {p_quantile:.2f} kPa")

p_quantile = brentq(lambda x: cdf_p(x) - 0.1, max_p_array.min(),
                    max_p_array.max() + 5*max_p_array.std())
print(f"Quantile 10% = {p_quantile:.2f} kPa")

In [ ]:
max_h_array = np.array(max_h)
kde_h = gaussian_kde(max_h_array, bw_method=0.2/max_h_array.std(ddof=1)*max_h_array.std(ddof=1))

total_h = quad(kde_h, 0, np.inf)[0]          # masse sur [0, +∞)
cdf_h = lambda x: quad(kde_h, 0, x)[0] / total_h

p_quantile = brentq(lambda x: cdf_h(x) - 0.9, max_h_array.min(),
                    max_h_array.max() + 5*max_h_array.std())
print(f"Quantile 90% = {p_quantile:.2f} m")

p_quantile = brentq(lambda x: cdf_h(x) - 0.5, max_h_array.min(),
                    max_h_array.max() + 5*max_h_array.std())
print(f"Quantile 50 % = {p_quantile:.2f} m")

In [ ]:
max_h_array.mean()

In [ ]:
q90 = np.quantile(max_p_array, 0.9)
q50 = np.quantile(max_p_array, 0.5)
q10 = np.quantile(max_p_array, 0.1)
print(f"Quantile 90 % = {q90:.2f} kPa")
print(f"Quantile 50 % = {q50:.2f} kPa")
print(f"Quantile 10 % = {q10:.2f} kPa")

In [ ]:
h90 = np.quantile(max_h_array, 0.9)
h50 = np.quantile(max_h_array, 0.5)
h10 = np.quantile(max_h_array, 0.1)
print(f"Quantile 90 % = {h90:.2f} m")
print(f"Quantile 50 % = {h50:.2f} m")
print(f"Quantile 10 % = {h10:.2f} m")

In [ ]:
len(max_p_array[max_p_array==0])/416

In [ ]:
len(max_p_array[max_p_array>30])/416